# Fock Matrix Construction and Schwarz Screening

This notebook covers two-electron integrals: the full ERI tensor, the compute-and-consume
pattern for building Fock matrices, and Schwarz screening to skip negligible integrals.

**What you'll learn:**
- Computing and verifying the ERI tensor $(\mu\nu|\lambda\sigma)$
- Building Coulomb ($J$) and exchange ($K$) matrices via `engine.compute(op, fock_builder)`
- Assembling the Fock matrix $F = H + J - \frac{1}{2}K$
- Using `SchwarzBounds` and `ScreeningOptions` for efficient integral screening
- How basis set size drives the $O(N^4)$ cost

In [ ]:
import numpy as np
import libaccint as lai

print(f"LibAccInt {lai.__version__}")
print(f"CUDA: {lai.has_cuda_backend()}")

## 1. Setup

Methane (CH$_4$) in the cc-pVDZ basis: a small but non-trivial system.

In [ ]:
# Tetrahedral CH4 in Bohr (C-H = 1.0870 Ang)
b = 1.185953834894377  # = (1.0870 / sqrt(3)) * 1.8897261246
atoms = [
    lai.Atom(6, [0.0, 0.0, 0.0]),     # C
    lai.Atom(1, [ b,  b,  b]),          # H
    lai.Atom(1, [ b, -b, -b]),          # H
    lai.Atom(1, [-b,  b, -b]),          # H
    lai.Atom(1, [-b, -b,  b]),          # H
]

basis = lai.basis_set("cc-pVDZ", atoms)
engine = lai.Engine(basis)
nbf = basis.n_basis_functions()

print(f"CH\u2084 / cc-pVDZ: {nbf} basis functions, {basis.n_shells()} shells")
print(f"GPU available: {engine.gpu_available()}")
print(f"ERI tensor size: {nbf}\u2074 = {nbf**4:,} elements ({nbf**4 * 8 / 1e6:.1f} MB)")

## 2. Full ERI Tensor

The electron repulsion integral $(\mu\nu|\lambda\sigma) = \int\int \mu(\mathbf{r}_1)\nu(\mathbf{r}_1) \frac{1}{r_{12}} \lambda(\mathbf{r}_2)\sigma(\mathbf{r}_2)\, d\mathbf{r}_1 d\mathbf{r}_2$

has 8-fold permutation symmetry in real bases.

In [ ]:
eri = engine.compute_eri_tensor()
print(f"ERI shape: {eri.shape}")

# Verify 8-fold permutation symmetry
# (mn|ls) = (nm|ls) = (mn|sl) = (nm|sl) = (ls|mn) = (sl|mn) = (ls|nm) = (sl|nm)
print("\nPermutation symmetry checks:")
print(f"  (mn|ls) = (nm|ls): {np.allclose(eri, eri.transpose(1,0,2,3))}")
print(f"  (mn|ls) = (mn|sl): {np.allclose(eri, eri.transpose(0,1,3,2))}")
print(f"  (mn|ls) = (ls|mn): {np.allclose(eri, eri.transpose(2,3,0,1))}")

## 3. FockBuilder: Compute-and-Consume Pattern

Instead of storing the $O(N^4)$ ERI tensor, `FockBuilder` accumulates the Coulomb and
exchange matrices on-the-fly. Call `engine.compute(op, fock_builder)` to compute all
two-electron integrals and consume them directly:

$$J_{\mu\nu} = \sum_{\lambda\sigma} D_{\lambda\sigma} (\mu\nu|\lambda\sigma), \qquad K_{\mu\nu} = \sum_{\lambda\sigma} D_{\lambda\sigma} (\mu\lambda|\nu\sigma)$$

In [ ]:
# Use identity density for a clean test
D = np.ascontiguousarray(np.eye(nbf), dtype=np.float64)

fock = lai.FockBuilder(nbf)
fock.set_density(D)

# engine.compute(op, consumer, hint) — the unified API for Fock builds.
# BackendHint.PreferGPU routes to the GPU when available, falling back to CPU.
engine.compute(lai.Operator.coulomb(), fock, lai.BackendHint.PreferGPU)

J = fock.get_coulomb_matrix()
K = fock.get_exchange_matrix()

print(f"J shape: {J.shape}, symmetric: {np.allclose(J, J.T)}")
print(f"K shape: {K.shape}, symmetric: {np.allclose(K, K.T)}")

In [ ]:
# Verify J and K against numpy einsum on the ERI tensor
J_ref = np.einsum('ls,mnls->mn', D, eri)
K_ref = np.einsum('ls,mlns->mn', D, eri)

print(f"J matches einsum: {np.allclose(J, J_ref)}, max |err|: {np.max(np.abs(J - J_ref)):.2e}")
print(f"K matches einsum: {np.allclose(K, K_ref)}, max |err|: {np.max(np.abs(K - K_ref)):.2e}")
assert np.allclose(J, J_ref) and np.allclose(K, K_ref), "J/K mismatch!"

## 4. Fock Matrix

The Fock matrix is $F = H_{\text{core}} + J - \frac{1}{2}K$ (using the Szabo convention where $D$ includes the factor of 2).

In [ ]:
H_core = engine.compute_core_hamiltonian(atoms)

# Via get_fock_matrix (exchange_fraction=0.5 for RHF)
F_api = fock.get_fock_matrix(H_core, 0.5)

# Manual assembly
F_manual = H_core + J - 0.5 * K

print(f"get_fock_matrix matches manual F = H + J - 0.5K: {np.allclose(F_api, F_manual)}")
print(f"Max |difference|: {np.max(np.abs(F_api - F_manual)):.2e}")

In [ ]:
# Convenience one-liner (creates a fresh FockBuilder internally)
F_conv = lai.build_fock(engine, D, H_core, exchange_fraction=0.5)

print(f"build_fock matches: {np.allclose(F_conv, F_manual)}")
print(f"Max |difference|: {np.max(np.abs(F_conv - F_manual)):.2e}")

## 5. Schwarz Screening

The Schwarz inequality bounds ERIs: $|(\mu\nu|\lambda\sigma)| \le Q_{\mu\nu} Q_{\lambda\sigma}$
where $Q_{\mu\nu} = \sqrt{(\mu\nu|\mu\nu)}$.

Quartets below the threshold are skipped, reducing cost from $O(N^4)$ toward $O(N^2)$ for large systems.

In [ ]:
bounds = lai.SchwarzBounds(basis)

print(f"Schwarz bounds for {bounds.n_shells()} shells")
print(f"Max bound: {bounds.max_bound():.6e}")

# Count passing quartets at different thresholds
n_total = bounds.n_shells()**4
print(f"\n{'Threshold':>12} {'Passing':>10} {'Fraction':>10}")
print("-" * 36)
for thresh in [1e-8, 1e-10, 1e-12, 1e-14]:
    n_pass = bounds.count_passing_quartets(thresh)
    frac = bounds.estimate_pass_fraction(thresh)
    print(f"{thresh:>12.0e} {n_pass:>10,} {frac:>10.1%}")

## 6. Screened Fock Build

Precompute Schwarz bounds in the engine, then use `compute_and_consume_screened()`.

In [ ]:
engine.precompute_schwarz_bounds()

opts = lai.ScreeningOptions.normal()  # threshold = 1e-12

fock_scr = lai.FockBuilder(nbf)
fock_scr.set_density(D)
engine.compute_and_consume_screened(lai.Operator.coulomb(), fock_scr, opts)

J_scr = fock_scr.get_coulomb_matrix()
K_scr = fock_scr.get_exchange_matrix()

# Compare to unscreened
print(f"Screened vs unscreened:")
print(f"  J max |error|: {np.max(np.abs(J_scr - J)):.2e}")
print(f"  K max |error|: {np.max(np.abs(K_scr - K)):.2e}")
print(f"  Both below threshold: {np.allclose(J_scr, J, atol=1e-10) and np.allclose(K_scr, K, atol=1e-10)}")

## 7. Basis Set Scaling

The number of two-electron integrals scales as $O(N_{\text{bf}}^4)$.
Screening becomes increasingly important with larger basis sets.

In [ ]:
print(f"{'Basis':<12} {'nbf':>5} {'nbf^4':>12} {'Schwarz pass (1e-12)':>22}")
print("-" * 55)
for name in ["STO-3G", "6-31G*", "cc-pVDZ"]:
    b = lai.basis_set(name, atoms)
    n = b.n_basis_functions()
    sb = lai.SchwarzBounds(b)
    frac = sb.estimate_pass_fraction(1e-12)
    print(f"{name:<12} {n:>5} {n**4:>12,} {frac:>21.1%}")

## Summary

| API | Purpose |
|-----|--------|
| `compute_eri_tensor()` | Full ERI tensor (small systems only) |
| `FockBuilder(nbf)` | On-the-fly J/K accumulator |
| `engine.compute(op, fock_builder)` | Compute ERIs and accumulate J, K |
| `get_fock_matrix(H, xfrac)` | Assemble $F = H + J - x \cdot K$ |
| `build_fock(engine, D, H)` | One-liner convenience |
| `SchwarzBounds(basis)` | Precompute Schwarz bounds |
| `ScreeningOptions.normal()` | Standard screening threshold |
| `compute_and_consume_screened(...)` | Screened Fock build |

**Next:** [04_hartree_fock.ipynb](04_hartree_fock.ipynb) — complete SCF calculation.